# Video 1: Combined top-down and back camera video trial examples

This notebook generates side-by-side video snippets from top-down and back camera views for selected trials, overlaying trial information and reward status.

*Note: You need to download the back videos and top-down videos for the session to work on.*

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
cd "/app/"

/app


/usr/local/lib/python3.9/dist-packages/IPython/core/magics/osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [3]:
%run env.py
%run run.py connect

2026-01-14 16:45:06,734::INFO::settings.py::Setting loglevel to INFO
2026-01-14 16:45:06,734::INFO::settings.py::Setting stores to {}
2026-01-14 16:45:06,735::INFO::settings.py::Setting database.misc.schema_prefix to 
2026-01-14 16:45:06,736::INFO::settings.py::Setting database.misc.create_tables to True
2026-01-14 16:45:06,736::INFO::settings.py::Setting enable_python_native_blobs to True
2026-01-14 16:45:06,736::INFO::settings.py::Setting database.host to 128.178.51.167:3309
2026-01-14 16:45:06,737::INFO::settings.py::Setting database.user to root
2026-01-14 16:45:06,737::INFO::settings.py::Setting database.password to simple


Connecting root@128.178.51.167:3309


OperationalError: (2003, "Can't connect to MySQL server on '128.178.51.167' (timed out)")

In [4]:
import pandas as pd
import numpy as np
import cv2

from vr4mice.schema.base_analysis import DataFrame
from vr4mice.schema import vr4mice
from vr4mice.analysis.analysis import get_rewarded

Connecting root@128.178.51.167:3309


2026-01-14 16:45:20,483::INFO::connection.py::Connected root@128.178.51.167:3309
2026-01-14 16:45:20,483::INFO::connection.py::Connected root@128.178.51.167:3309


In [ ]:
def get_occlusion_percentage(aperture_size):
    if aperture_size == 3.:
        return 0
    elif aperture_size == 8.48:
        return 72
    elif aperture_size == 6.:
        return 35
    elif aperture_size == 12.:
        return 99
    elif aperture_size == 4.3 or aperture_size == 4.2:
        return 10
    else:
        return None

def generate_side_by_side_snippets(df, top_path, back_path, output_name, speed_factor=0.4, display_reward=False):
    try:
        cap_top = cv2.VideoCapture(top_path)
        cap_back = cv2.VideoCapture(back_path)
        
        master_fps = 100
        output_fps = master_fps * speed_factor 
        
        w1 = int(cap_top.get(cv2.CAP_PROP_FRAME_WIDTH))
        h1 = int(cap_top.get(cv2.CAP_PROP_FRAME_HEIGHT))
        w2 = int(cap_back.get(cv2.CAP_PROP_FRAME_WIDTH))
        h2 = int(cap_back.get(cv2.CAP_PROP_FRAME_HEIGHT))
        
        if w1 <= 0 or h1 <=0 or w2 <= 0 or h2 <= 0:
            cap_top.release()
            cap_back.release()
            raise ValueError(f"Invalid video dimensions: top ({w1}x{h1}), back ({w2}x{h2})")

        # Resize logic for vertical alignment
        scale_factor = h1 / h2
        new_w2 = int(w2 * scale_factor)
        
        fourcc = cv2.VideoWriter_fourcc(*'mp4v')
        out = cv2.VideoWriter(output_name, fourcc, output_fps, (w1 + new_w2, h1))

        for _, row in df.iterrows():
            # Seek to specific aligned frames
            cap_top.set(cv2.CAP_PROP_POS_FRAMES, int(row['top_frames']))
            cap_back.set(cv2.CAP_PROP_POS_FRAMES, int(row['back_frames']))
            
            ret1, frame1 = cap_top.read()
            ret2, frame2 = cap_back.read()
            
            if ret1 and ret2:
                frame1 = cv2.rotate(frame1, cv2.ROTATE_180)
                frame2_resized = cv2.resize(frame2, (new_w2, h1))
                combined = np.hstack((frame1, frame2_resized))
                
                font = cv2.FONT_HERSHEY_SIMPLEX
                font_scale = 0.6
                thickness = 1
                
                # Overlay trial info and individual timestamps
                if display_reward:
                    reward_info = f"SUCCESS (rewarded)" if int(row['rewarded_trial']) else "FAILURE (unrewarded)"
                    cv2.putText(combined, reward_info, (20, 30), 
                                font, font_scale, (0, 255, 0) if row['rewarded_trial'] > 0 else (0, 0, 255), thickness)
                
                aperture_info = f"Occlusion: {get_occlusion_percentage(row['aperture'])}%"
                cv2.putText(combined, aperture_info, (20, 60), 
                            font, font_scale, (255, 255, 255), thickness)
                
                info = f"Trial: {int(row['trial'])}"
                cv2.putText(combined, info, (20, h1 - 20), 
                            font, font_scale, (255, 255, 255), thickness)
                
                out.write(combined)
    
    finally:
        cap_top.release()
        cap_back.release()
        out.release()

In [6]:
def get_aligned_videos(dataset):
    # Back camera related files
    cam_back_video_path = f"/app/back_videos/CamBack_{dataset}.avi"
    cam_back_timesteps_path = f"/app/back_videos/TIMESTAMPS_CamBack_{dataset}.npy"

    # Top-down camera related files
    cam_topdown_video_path = f"/app/back_videos/Imagingsource_{dataset}_VIDEO.avi"
    cam_topdown_times_path = f"/app/back_videos/Imagingsource_{dataset}_TS.npy"
    
    with open(cam_back_timesteps_path, 'rb') as f:
        back_timesteps_data = np.load(f)
    with open(cam_topdown_times_path, 'rb') as f:
        topdown_times_data = np.load(f)
        
    start_time_game = (vr4mice.State() & {"dataset": dataset}).fetch("start_time")[0]
    
    topdown_timesteps = topdown_times_data - start_time_game
    back_timesteps = back_timesteps_data - start_time_game
    
    top_df = pd.DataFrame({
        "time_ms": topdown_timesteps,
        "top_frames": range(len(topdown_timesteps))
    })
    back_df = pd.DataFrame({
        "time_ms": back_timesteps,
        "back_frames": range(len(back_timesteps))
    })
    
    aligned_df = pd.merge_asof(
        top_df.sort_values('time_ms'),
        back_df.sort_values('time_ms'),
        on='time_ms',
        direction='nearest'
    )
    
    return aligned_df, cam_topdown_video_path, cam_back_video_path

In [7]:
# # Allow up to 0.5 seconds (500 ms) difference when aligning behavioral timestamps
# (`step_time`) with video timestamps (`time_ms`). This tolerance is tight enough
# to prevent mismatching unrelated events while accounting for small timing jitter.
time_tolerance = 0.5

## Dual occlusion example: "Pheasant_2024-11-08_15-2"

In [8]:
dual_dataset = "Pheasant_2024-08-15_2"

In [9]:
aligned_df, cam_topdown_video_path, cam_back_video_path = get_aligned_videos(dual_dataset)

In [10]:
df = DataFrame().get_data(
    key={"dataset": dual_dataset},
    columns=[
        "dataset",
        "reward",
        "trial",
        "aperture",
        "iti",
        "trial_left_choice",
        "step",
        "step_time",
        "time_elapsed",
        "x",
        "y",
    ],
)

In [11]:
total_df = pd.merge_asof(df.sort_values('step_time'),
                         aligned_df.sort_values('time_ms'),
                            left_on='step_time',
                            right_on='time_ms',
                            direction='nearest', tolerance=time_tolerance)

total_df["rewarded_trial"] = get_rewarded(total_df)

### Set of both rewarded and failed trials

In [12]:
selected_trials = np.arange(11, 250, 15)
selected_df = total_df[(total_df["iti"]<1) & (total_df["trial"].isin(selected_trials))]

In [13]:
generate_side_by_side_snippets(selected_df, cam_topdown_video_path, 
                               cam_back_video_path, f'/app/back_videos/combined_top_back_{dual_dataset}.mp4',
                               display_reward=True)

### Rewarded trials

In [14]:
selected_trials = np.arange(11, 250, 10)
selected_df = total_df[(total_df["iti"]<1) & (total_df["rewarded_trial"]==1) & (total_df["trial"].isin(selected_trials))]

In [15]:
generate_side_by_side_snippets(selected_df, cam_topdown_video_path, 
                               cam_back_video_path, 
                               f'/app/back_videos/combined_top_back_rewarded_{dual_dataset}.mp4',
                               display_reward=False)

### Failed trials

In [16]:
selected_trials = np.arange(11, 250, 10)
selected_df = total_df[(total_df["iti"]<1) & (total_df["rewarded_trial"]==0) & (total_df["trial"].isin(selected_trials))]

In [17]:
generate_side_by_side_snippets(selected_df, cam_topdown_video_path, cam_back_video_path, 
                               f'/app/back_videos/combined_top_back_failed_{dual_dataset}.mp4',
                               display_reward=False)

## Multi occlusion video example: Nightingale_2024-08-16_1

In [18]:
multi_dataset = "Nightingale_2024-08-16_1"

In [19]:
aligned_df, cam_topdown_video_path, cam_back_video_path = get_aligned_videos(multi_dataset)

In [20]:
df = DataFrame().get_data(
    key={"dataset": multi_dataset},
    columns=[
        "dataset",
        "reward",
        "trial",
        "aperture",
        "iti",
        "trial_left_choice",
        "step",
        "step_time",
        "time_elapsed",
        "x",
        "y",
    ],
)

In [21]:
total_df = pd.merge_asof(df.sort_values('step_time'),
                         aligned_df.sort_values('time_ms'),
                            left_on='step_time',
                            right_on='time_ms',
                            direction='nearest', tolerance=time_tolerance)

total_df["rewarded_trial"] = get_rewarded(total_df)

### Set of both rewarded and failed trials

In [22]:
selected_trials = np.arange(12, 250, 15)
selected_df = total_df[(total_df["iti"]<1) & (total_df["trial"].isin(selected_trials))]

In [23]:
generate_side_by_side_snippets(selected_df, cam_topdown_video_path, cam_back_video_path, 
                               f'/app/back_videos/combined_top_back_{multi_dataset}.mp4',
                               display_reward=True)

### Rewarded trials

In [24]:
selected_trials = np.arange(12, 250, 10)
selected_df = total_df[(total_df["iti"]<1) & (total_df["rewarded_trial"]==1) & (total_df["trial"].isin(selected_trials))]

In [25]:
generate_side_by_side_snippets(selected_df, cam_topdown_video_path, cam_back_video_path, 
                               f'/app/back_videos/combined_top_back_rewarded_{multi_dataset}.mp4')

### Failed trials

In [26]:
selected_trials = np.arange(12, 250, 10)
selected_df = total_df[(total_df["iti"]<1) & (total_df["rewarded_trial"]==0) & (total_df["trial"].isin(selected_trials))]

In [27]:
generate_side_by_side_snippets(selected_df, cam_topdown_video_path, cam_back_video_path, 
                               f'/app/back_videos/combined_top_back_failed_{multi_dataset}.mp4')